# ⚔️ Arcane Clash: Playstyle Classifier & Fighting Style Predictor
### Complete Isolated Training, Testing & Interactive Inference Pipeline

This notebook is an **end-to-end, isolated pipeline** for training, evaluating, and predicting player **Fighting Styles / Playstyle Archetypes** in *Arcane Clash*.

---

### 🎯 Key Capabilities:
1. **100% Self-Contained:** Can run locally or on Google Colab without broken file paths or external script dependencies.
2. **Flexible Data Pipeline:** Automatically detects local match logs (if present) OR generates a balanced synthetic dataset of simulated bot matches across all 4 archetypes.
3. **49-Dimensional Feature Extraction:** Extracts action distributions, combat efficiency, resource management, counter-play telegraph response, clutch behaviors, and sequential bigrams.
4. **PyTorch MLP Architecture:** Deep learning classifier with Kaiming initialization, dropout regularization, and early stopping.
5. **Evaluation Suite:** Confusion matrix heatmap, per-class Precision/Recall/F1-score, and training trajectory plots.
6. **Interactive Fighting Style Predictor:** Provide your battle actions, combat turns, or tactical traits to instantly predict your fighting style with confidence percentages, probability visualizations, and tactical profile breakdowns!


In [ ]:
# ==============================================================================
# 1. SETUP & ENVIRONMENT
# ==============================================================================
# Uncomment and run the following line if running on Google Colab or fresh environment:
# !pip install torch torchvision numpy pandas scikit-learn matplotlib seaborn joblib

import os
import sys

# Ensure UTF-8 output encoding across Windows consoles & Jupyter kernels
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import math
import json
import random
import time
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns  # type: ignore
except ImportError:
    sns = None

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Set reproducibility seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 PyTorch Device: {device}")
print(f"📦 PyTorch Version: {torch.__version__}")


In [ ]:
# ==============================================================================
# 2. FIGHTING STYLE ARCHETYPES & GAME DOMAIN
# ==============================================================================

PERSONA_LABELS = ["Aggressive", "Defensive", "Optimal", "Random"]
LABEL_TO_IDX   = {label: idx for idx, label in enumerate(PERSONA_LABELS)}
IDX_TO_LABEL   = {idx: label for label, idx in LABEL_TO_IDX.items()}
NUM_CLASSES    = len(PERSONA_LABELS)

# Rich fighting style archetype metadata
PERSONA_PROFILES = {
    "Aggressive": {
        "title": "Relentless Berserker",
        "badge_color": "#FF4757",
        "description": "Favors immediate high-damage spells and relentless pressure, delaying healing until critical health.",
        "icon": "⚔️",
        "key_traits": ["High Fireball & Fire Spell usage", "Low heal HP threshold (<20%)", "High damage output", "Low counter-play focus"]
    },
    "Defensive": {
        "title": "Tactical Guardian",
        "badge_color": "#2ED573",
        "description": "Anticipates boss telegraphs with precise soft-counters and maintains high health reserves via proactive recovery.",
        "icon": "🛡️",
        "key_traits": ["High counter-play rate (>60%)", "Frequent Rest & Heal", "High heal trigger HP (>50%)", "Consistent resource buffer"]
    },
    "Optimal": {
        "title": "Grand Strategist",
        "badge_color": "#9B59B6",
        "description": "Methodically balances resource thresholds (XP>80, Power>50) before unleashing high-value bursts and tactical counters.",
        "icon": "⚡",
        "key_traits": ["Balanced spell rotation", "High XP/Power management", "Adapts counter-play in phase 2", "Optimal damage-trade ratio"]
    },
    "Random": {
        "title": "Wildcard Sorcerer",
        "badge_color": "#FFA502",
        "description": "Exhibits unpredictable spell variety and high action entropy without adhering to rigid tactical loops.",
        "icon": "🎲",
        "key_traits": ["High Shannon action entropy", "Even spread of all actions", "Low repeat rate", "Uncorrelated counter timing"]
    }
}

# Recognized Player Actions
KNOWN_ACTIONS = [
    "Quick Slash",
    "Fireball",
    "Water Arrow",
    "Water Spell",
    "Fire Spell",
    "Heal",
    "Rest & Recharge",
]

_ACTION_KEYS = [
    "quick_slash",
    "fireball",
    "water_arrow",
    "water_spell",
    "fire_spell",
    "heal",
    "rest_and_recharge",
]

_LABEL_TO_KEY = {label: key for label, key in zip(KNOWN_ACTIONS, _ACTION_KEYS)}
# Also add lowercase aliases
for k in _ACTION_KEYS:
    _LABEL_TO_KEY[k] = k
_LABEL_TO_KEY["rest"] = "rest_and_recharge"

HERO_MAX_HP = {
    "Fire Wizard": 120,
    "Lightning Mage": 100,
    "Wanderer Magician": 110,
    "Wanderer": 110,
}
DEFAULT_MAX_HP = 110

# Canonical 49 feature columns aligned with production model
FEATURE_COLUMNS = [
    'pct_quick_slash', 'pct_fireball', 'pct_water_arrow', 'pct_water_spell',
    'pct_fire_spell', 'pct_heal', 'pct_rest_and_recharge', 'action_entropy',
    'opening_is_quick_slash', 'opening_is_fireball', 'opening_is_water_arrow',
    'opening_is_water_spell', 'opening_is_fire_spell', 'opening_is_heal',
    'opening_is_rest_and_recharge', 'avg_power_reserve', 'avg_xp_reserve',
    'power_reserve_std', 'xp_reserve_std', 'min_power_reached', 'min_xp_reached',
    'resource_floor_events', 'total_damage_dealt', 'total_damage_taken',
    'damage_trade_ratio', 'miss_rate', 'miss_rate_by_risky_spell',
    'counter_play_rate', 'counter_play_rate_first_half', 'counter_play_rate_second_half',
    'counter_play_improvement', 'heal_avg_hp_pct_trigger', 'clutch_action_entropy',
    'fire_spell_usage_rate', 'comeback_flag', 'avg_reaction_time_ms',
    'median_reaction_time_ms', 'reaction_time_std', 'reaction_time_cv',
    'slow_decision_rate', 'status_effect_uptime_rate', 'turns_to_clear_weakened',
    'repeat_action_rate', 'post_damage_defensive_rate',
    'action_bigram_rest_and_recharge_rest_and_recharge',
    'action_bigram_water_arrow_rest_and_recharge',
    'action_bigram_rest_and_recharge_heal',
    'action_bigram_rest_and_recharge_water_arrow',
    'action_bigram_heal_rest_and_recharge'
]

print(f"✅ Configured {NUM_CLASSES} Playstyle Archetypes: {', '.join(PERSONA_LABELS)}")
print(f"✅ Canonical Feature Dimension: {len(FEATURE_COLUMNS)} input features")


In [ ]:
# ==============================================================================
# 3. STANDALONE 49-DIMENSIONAL FEATURE EXTRACTION ENGINE
# ==============================================================================

def _safe_div(numerator: float, denominator: float, default: float = 0.0) -> float:
    return float(numerator / denominator) if denominator != 0 else default

def _shannon_entropy(counts: Dict[str, int]) -> float:
    total = sum(counts.values())
    if total == 0:
        return 0.0
    probs = [c / total for c in counts.values() if c > 0]
    return float(-sum(p * math.log2(p) for p in probs))

def extract_match_features(match_json: Dict[str, Any]) -> Dict[str, Any]:
    """
    Converts a match record (turns + summary) into the exact 49 numeric features.
    Gracefully handles partial dictionaries and default values.
    """
    summary = match_json.get("summary", {})
    turns   = match_json.get("turns", [])
    
    hero_class = summary.get("hero_class", "Wanderer Magician")
    max_hp = HERO_MAX_HP.get(hero_class, DEFAULT_MAX_HP)
    n_turns = len(turns)
    total_turns = max(n_turns, 1)
    
    feats: Dict[str, Any] = {}
    
    # --- GROUP A: Action Distribution & Opening Action ---
    action_counts: Dict[str, int] = {k: 0 for k in _ACTION_KEYS}
    for t in turns:
        raw_act = t.get("action_chosen", "")
        key = _LABEL_TO_KEY.get(raw_act)
        if key and key in action_counts:
            action_counts[key] += 1
            
    for k in _ACTION_KEYS:
        feats[f"pct_{k}"] = _safe_div(action_counts[k], total_turns)
    feats["action_entropy"] = _shannon_entropy(action_counts)
    
    opening_turn = turns[0] if turns else {}
    opening_key = _LABEL_TO_KEY.get(opening_turn.get("action_chosen", ""), "")
    for k in _ACTION_KEYS:
        feats[f"opening_is_{k}"] = 1.0 if k == opening_key else 0.0

    # --- GROUP B: Resource Management ---
    power_vals = [t.get("player_power_before", 50) for t in turns] or [50.0]
    xp_vals    = [t.get("player_xp_before",    50) for t in turns] or [50.0]
    
    feats["avg_power_reserve"] = float(np.mean(power_vals))
    feats["avg_xp_reserve"]    = float(np.mean(xp_vals))
    feats["power_reserve_std"] = float(np.std(power_vals))
    feats["xp_reserve_std"]    = float(np.std(xp_vals))
    feats["min_power_reached"] = float(min(power_vals))
    feats["min_xp_reached"]    = float(min(xp_vals))
    feats["resource_floor_events"] = float(sum(
        1 for t in turns if t.get("player_power_before", 50) < 10 or t.get("player_xp_before", 50) < 10
    ))

    # --- GROUP C: Combat Efficiency ---
    dealt = sum(t.get("damage_dealt", 0) for t in turns)
    taken = sum(t.get("damage_taken", 0) for t in turns)
    feats["total_damage_dealt"] = float(dealt)
    feats["total_damage_taken"] = float(taken)
    feats["damage_trade_ratio"] = _safe_div(dealt, max(taken, 1))
    
    misses = sum(1 for t in turns if not t.get("hit_success", True))
    feats["miss_rate"] = _safe_div(misses, total_turns)
    
    risky_turns = [t for t in turns if t.get("action_chosen", "") in ("Fireball", "Fire Spell", "fireball", "fire_spell")]
    risky_misses = sum(1 for t in risky_turns if not t.get("hit_success", True))
    feats["miss_rate_by_risky_spell"] = _safe_div(risky_misses, max(len(risky_turns), 1))

    # --- GROUP D: Threat Response & Counter-Play ---
    half = total_turns / 2.0
    first_half  = [t for t in turns if t.get("turn_number", 0) <= half]
    second_half = [t for t in turns if t.get("turn_number", 0) > half]
    
    cnt_all = sum(1 for t in turns if t.get("action_matched_counter", False))
    cnt_1st = sum(1 for t in first_half if t.get("action_matched_counter", False))
    cnt_2nd = sum(1 for t in second_half if t.get("action_matched_counter", False))
    
    feats["counter_play_rate"]             = _safe_div(cnt_all, total_turns)
    feats["counter_play_rate_first_half"]  = _safe_div(cnt_1st, max(len(first_half), 1))
    feats["counter_play_rate_second_half"] = _safe_div(cnt_2nd, max(len(second_half), 1))
    feats["counter_play_improvement"]      = feats["counter_play_rate_second_half"] - feats["counter_play_rate_first_half"]

    # --- GROUP E: Risk & Clutch Behavior ---
    heal_turns = [t for t in turns if _LABEL_TO_KEY.get(t.get("action_chosen", "")) == "heal"]
    if heal_turns:
        feats["heal_avg_hp_pct_trigger"] = float(np.mean([t.get("player_hp_before", max_hp) / max_hp for t in heal_turns]))
    else:
        feats["heal_avg_hp_pct_trigger"] = 0.0
        
    clutch_turns = [t for t in turns if t.get("player_hp_before", max_hp) / max_hp < 0.30]
    if clutch_turns:
        c_counts = {k: 0 for k in _ACTION_KEYS}
        for t in clutch_turns:
            k = _LABEL_TO_KEY.get(t.get("action_chosen", ""))
            if k in c_counts: c_counts[k] += 1
        feats["clutch_action_entropy"] = _shannon_entropy(c_counts)
    else:
        feats["clutch_action_entropy"] = 0.0
        
    feats["fire_spell_usage_rate"] = feats["pct_fire_spell"]
    feats["comeback_flag"] = 1.0 if summary.get("comeback_flag", False) else 0.0

    # --- GROUP F: Decision Timing ---
    reaction_times = [t.get("time_taken_to_decide", 1200) for t in turns] or [1200.0]
    mean_rt = float(np.mean(reaction_times))
    std_rt  = float(np.std(reaction_times))
    feats["avg_reaction_time_ms"]    = mean_rt
    feats["median_reaction_time_ms"] = float(np.median(reaction_times))
    feats["reaction_time_std"]       = std_rt
    feats["reaction_time_cv"]        = _safe_div(std_rt, max(mean_rt, 1e-6))
    feats["slow_decision_rate"]      = _safe_div(sum(1 for rt in reaction_times if rt > 15000), total_turns)

    # --- GROUP G: Status Effects ---
    status_active = sum(1 for t in turns if len(t.get("status_effects_active", [])) > 0)
    feats["status_effect_uptime_rate"] = _safe_div(status_active, total_turns)
    feats["turns_to_clear_weakened"]   = float(summary.get("turns_to_clear_weakened", 0.0))

    # --- GROUP H: Sequential & Bigram Transitions ---
    seq = [_LABEL_TO_KEY.get(t.get("action_chosen", ""), "") for t in turns]
    repeats = sum(1 for i in range(1, len(seq)) if seq[i] == seq[i-1] and seq[i])
    feats["repeat_action_rate"] = _safe_div(repeats, max(len(seq) - 1, 1))
    
    post_dmg_tot = 0
    post_dmg_def = 0
    for i in range(len(turns) - 1):
        if turns[i].get("damage_taken", 0) > 20:
            post_dmg_tot += 1
            next_act = _LABEL_TO_KEY.get(turns[i+1].get("action_chosen", ""), "")
            if next_act in ("rest_and_recharge", "heal"):
                post_dmg_def += 1
    feats["post_damage_defensive_rate"] = _safe_div(post_dmg_def, max(post_dmg_tot, 1))

    # Specified Top-5 Bigrams
    n_bigrams = max(len(seq) - 1, 1)
    bigram_counts = {}
    for i in range(len(seq) - 1):
        if seq[i] and seq[i+1]:
            bg_key = f"{seq[i]}_{seq[i+1]}"
            bigram_counts[bg_key] = bigram_counts.get(bg_key, 0) + 1
            
    feats["action_bigram_rest_and_recharge_rest_and_recharge"] = _safe_div(bigram_counts.get("rest_and_recharge_rest_and_recharge", 0), n_bigrams)
    feats["action_bigram_water_arrow_rest_and_recharge"]         = _safe_div(bigram_counts.get("water_arrow_rest_and_recharge", 0), n_bigrams)
    feats["action_bigram_rest_and_recharge_heal"]               = _safe_div(bigram_counts.get("rest_and_recharge_heal", 0), n_bigrams)
    feats["action_bigram_rest_and_recharge_water_arrow"]        = _safe_div(bigram_counts.get("rest_and_recharge_water_arrow", 0), n_bigrams)
    feats["action_bigram_heal_rest_and_recharge"]               = _safe_div(bigram_counts.get("heal_rest_and_recharge", 0), n_bigrams)

    return feats

print("✅ Feature extraction engine ready!")


In [ ]:
# ==============================================================================
# 4. SELF-CONTAINED MATCH GENERATOR (ISOLATED DATA GENERATION)
# ==============================================================================
# If local match logs are found in ./logs or ../logs, we can load them.
# Otherwise, we generate synthetic bot matches for all 4 archetypes on the fly!

def simulate_match(persona: str, match_id: int) -> Dict[str, Any]:
    """
    Simulates a battle between a specific persona bot and a boss.
    Adheres to the combat rules and generates a rich match JSON record.
    """
    hero_class = random.choice(["Fire Wizard", "Lightning Mage", "Wanderer Magician"])
    max_hp = HERO_MAX_HP[hero_class]
    hero_hp = max_hp
    power = random.randint(40, 60)
    xp = random.randint(30, 50)
    
    boss_max_hp = random.choice([120, 150, 180])
    boss_hp = boss_max_hp
    
    turns = []
    num_turns = random.randint(8, 16)
    last_action = ""
    
    for turn_no in range(1, num_turns + 1):
        boss_telegraph = random.choice(["fireball", "fire_spell", "quick_slash", "rest", "water_spell"])
        
        # Persona decision logic
        if persona == "Aggressive":
            # Favors raw damage; heals only under 20% HP
            if hero_hp / max_hp < 0.20 and random.random() < 0.5:
                action = "Heal"
            elif power >= 30 and random.random() < 0.7:
                action = random.choice(["Fireball", "Fire Spell"])
            else:
                action = random.choice(["Quick Slash", "Water Arrow", "Fireball"])
                
        elif persona == "Defensive":
            # Counters boss telegraphs; restores HP and power proactively
            if hero_hp / max_hp < 0.55 and random.random() < 0.6:
                action = "Heal"
            elif boss_telegraph in ("fireball", "fire_spell", "water_spell"):
                action = random.choice(["Rest & Recharge", "Water Arrow"])
            elif power < 30:
                action = "Rest & Recharge"
            else:
                action = random.choice(["Water Arrow", "Quick Slash"])
                
        elif persona == "Optimal":
            # Methodical: builds resources, counters high threats, combos at peak resources
            if hero_hp / max_hp < 0.35:
                action = "Heal"
            elif boss_telegraph in ("fireball", "fire_spell") and random.random() < 0.85:
                action = "Water Arrow"
            elif power > 60 and xp > 50:
                action = random.choice(["Fire Spell", "Fireball"])
            elif power < 25:
                action = "Rest & Recharge"
            else:
                action = random.choice(["Quick Slash", "Water Arrow"])
                
        else: # Random (Wildcard)
            action = random.choice(KNOWN_ACTIONS)
            
        # Determine counter match
        is_counter = False
        if boss_telegraph in ("fireball", "fire_spell", "water_spell") and action in ("Rest & Recharge", "Water Arrow"):
            is_counter = True
        elif boss_telegraph == "rest" and action in ("Quick Slash", "Fireball", "Fire Spell"):
            is_counter = True
            
        # Combat math simulation
        hit_success = random.random() > (0.15 if action in ("Fireball", "Fire Spell") else 0.05)
        dmg_dealt = 0
        if hit_success:
            if action in ("Fireball", "Fire Spell"): dmg_dealt = random.randint(22, 38)
            elif action in ("Quick Slash", "Water Arrow", "Water Spell"): dmg_dealt = random.randint(12, 22)
            
        dmg_taken = random.randint(10, 25) if not is_counter else random.randint(2, 10)
        
        # Decision times (Aggressive = fast, Optimal = thoughtful, Random = erratic)
        if persona == "Aggressive": rt = random.gauss(800, 200)
        elif persona == "Defensive": rt = random.gauss(1500, 300)
        elif persona == "Optimal": rt = random.gauss(2200, 400)
        else: rt = random.uniform(500, 8000)
        rt = max(300, min(20000, rt))
        
        # Resource adjustments
        if action == "Rest & Recharge":
            power = min(100, power + 35)
        elif action in ("Fireball", "Fire Spell"):
            power = max(0, power - 25)
        elif action == "Heal":
            power = max(0, power - 15)
            hero_hp = min(max_hp, hero_hp + 30)
            
        turn_record = {
            "turn_number": turn_no,
            "action_chosen": action,
            "player_hp_before": hero_hp,
            "player_power_before": power,
            "player_xp_before": xp,
            "damage_dealt": dmg_dealt,
            "damage_taken": dmg_taken,
            "hit_success": hit_success,
            "action_matched_counter": is_counter,
            "time_taken_to_decide": rt,
            "status_effects_active": ["Weakened"] if random.random() < 0.2 else []
        }
        turns.append(turn_record)
        
        hero_hp = max(0, hero_hp - dmg_taken)
        boss_hp = max(0, boss_hp - dmg_dealt)
        last_action = action
        if hero_hp <= 0 or boss_hp <= 0:
            break
            
    summary = {
        "match_id": f"sim_match_{match_id}",
        "player_id": f"bot_{persona.lower()}_{match_id}",
        "persona_label": persona,
        "hero_class": hero_class,
        "outcome": "win" if hero_hp > 0 else "loss",
        "comeback_flag": hero_hp > 0 and any(t["player_hp_before"] < max_hp * 0.25 for t in turns)
    }
    
    return {"summary": summary, "turns": turns}

def load_or_generate_dataset(n_matches_per_persona: int = 150) -> pd.DataFrame:
    """
    Searches for local match logs first; if fewer than 40 found, generates
    a clean balanced dataset of simulated matches.
    """
    records = []
    
    # Check potential local log directories
    candidate_dirs = ["logs", "../logs", "model/logs", "../model/logs"]
    for c_dir in candidate_dirs:
        if os.path.exists(c_dir):
            for fname in os.listdir(c_dir):
                if fname.startswith("match_") and fname.endswith(".json"):
                    try:
                        with open(os.path.join(c_dir, fname), "r", encoding="utf-8") as f:
                            data = json.load(f)
                            p_id = data.get("summary", {}).get("player_id", "")
                            # Check persona from player_id
                            label = None
                            for p in PERSONA_LABELS:
                                if p.lower() in p_id.lower():
                                    label = p
                                    break
                            if label:
                                row: Dict[str, Any] = dict(extract_match_features(data))
                                row["persona_label"] = label
                                records.append(row)
                    except Exception:
                        pass
                        
    if len(records) >= 80:
        print(f"📂 Loaded {len(records)} existing match logs from local disk.")
        return pd.DataFrame(records)
        
    print(f"⚡ Generating fresh balanced dataset ({n_matches_per_persona} matches per persona = {n_matches_per_persona * 4} total)...")
    records = []
    match_cnt = 0
    for persona in PERSONA_LABELS:
        for _ in range(n_matches_per_persona):
            match_cnt += 1
            sim_match = simulate_match(persona, match_cnt)
            row: Dict[str, Any] = dict(extract_match_features(sim_match))
            row["persona_label"] = persona
            records.append(row)
            
    df = pd.DataFrame(records)
    print(f"✅ Generated dataset: {len(df)} samples across {df['persona_label'].nunique()} classes.")
    return df

df_matches = load_or_generate_dataset(n_matches_per_persona=150)
df_matches["persona_label"].value_counts().plot(kind="bar", color=["#FF4757", "#2ED573", "#9B59B6", "#FFA502"], figsize=(7, 3.5), title="Class Distribution")
plt.xlabel("Fighting Style Archetype")
plt.ylabel("Match Count")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# ==============================================================================
# 5. DATA PREPARATION: TRAIN / VAL / TEST SPLIT & NORMALIZATION
# ==============================================================================

# Ensure all 49 feature columns exist (fill missing with 0.0)
for col in FEATURE_COLUMNS:
    if col not in df_matches.columns:
        df_matches[col] = 0.0

X = df_matches[FEATURE_COLUMNS].values.astype(np.float32)
y = df_matches["persona_label"].map(LABEL_TO_IDX).values.astype(np.int64)

# Stratified 70% Train, 15% Validation, 15% Test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Standardize features (Scaler is strictly fitted on TRAIN only to prevent leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)

# Save scaler for inference
joblib.dump(scaler, "feature_scaler.joblib")
print("💾 Fitted feature scaler saved as 'feature_scaler.joblib'.")

# Create PyTorch DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_val_s), torch.tensor(y_val)), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test_s), torch.tensor(y_test)), batch_size=BATCH_SIZE, shuffle=False)

print(f"📊 Dataset Split Summary:")
print(f"   - Training samples   : {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"   - Validation samples : {len(X_val)}   ({len(X_val)/len(X):.1%})")
print(f"   - Test samples       : {len(X_test)}  ({len(X_test)/len(X):.1%})")
print(f"   - Input Dimension    : {X.shape[1]} features")


In [ ]:
# ==============================================================================
# 6. PYTORCH MODEL ARCHITECTURE: PLAYSTYLE CLASSIFIER
# ==============================================================================

class PlaystyleClassifier(nn.Module):
    """
    A deep MLP classifier that maps a 49-dimensional playstyle feature vector
    to the 4 fighting style classes (Aggressive, Defensive, Optimal, Random).
    """
    def __init__(self, input_dim: int = 49, hidden_dim: int = 32, num_classes: int = 4, dropout: float = 0.2):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        """Kaiming He initialization for ReLU activations."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return self.forward(x).argmax(dim=-1)

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return torch.softmax(self.forward(x), dim=-1)

model = PlaystyleClassifier(input_dim=len(FEATURE_COLUMNS), hidden_dim=32, num_classes=NUM_CLASSES).to(device)
print(model)
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# ==============================================================================
# 7. TRAINING & VALIDATION LOOP WITH EARLY STOPPING
# ==============================================================================

EPOCHS = 100
LEARNING_RATE = 0.003
WEIGHT_DECAY = 1e-4
PATIENCE = 15

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [],  "val_acc": []
}

best_val_loss = float("inf")
best_state = None
patience_counter = 0

print("⏳ Starting model training...")
for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        outputs = model(X_b)
        loss = criterion(outputs, y_b)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * len(y_b)
        preds = outputs.argmax(dim=-1)
        correct += (preds == y_b).sum().item()
        total += len(y_b)
        
    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validation Phase ---
    model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            outputs = model(X_b)
            loss = criterion(outputs, y_b)
            val_running_loss += loss.item() * len(y_b)
            preds = outputs.argmax(dim=-1)
            val_correct += (preds == y_b).sum().item()
            val_total += len(y_b)
            
    val_loss = val_running_loss / val_total
    val_acc  = val_correct / val_total
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # Check Early Stopping & Save Best Weights
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1 or patience_counter >= PATIENCE:
        print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")

    if patience_counter >= PATIENCE:
        print(f"🛑 Early stopping triggered at epoch {epoch}! Restoring best model weights...")
        break

# Restore best checkpoint
if best_state is not None:
    model.load_state_dict(best_state)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(history["train_loss"], label="Train Loss", color="#FF4757", linewidth=2)
ax1.plot(history["val_loss"], label="Val Loss", color="#1E90FF", linewidth=2, linestyle="--")
ax1.set_title("Cross-Entropy Loss Curve", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.plot(history["train_acc"], label="Train Accuracy", color="#2ED573", linewidth=2)
ax2.plot(history["val_acc"], label="Val Accuracy", color="#9B59B6", linewidth=2, linestyle="--")
ax2.set_title("Classification Accuracy Curve", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# 8. TEST SET EVALUATION & MODEL CHECKPOINT EXPORT
# ==============================================================================

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model(X_b).argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_b.numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)
test_acc    = (all_preds == all_targets).mean()

print(f"🎯 Final Test Accuracy: {test_acc:.4f} ({test_acc:.1%})\n")
print("📋 Detailed Classification Report:")
print(classification_report(all_targets, all_preds, target_names=PERSONA_LABELS, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6.5, 5))
if sns is not None:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
                xticklabels=PERSONA_LABELS, yticklabels=PERSONA_LABELS, cbar=False)
else:
    plt.imshow(cm, cmap="Purples", interpolation="nearest")
    for i in range(len(PERSONA_LABELS)):
        for j in range(len(PERSONA_LABELS)):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black" if cm[i, j] < cm.max()/2 else "white", fontweight="bold")
    plt.xticks(range(len(PERSONA_LABELS)), PERSONA_LABELS, rotation=30)
    plt.yticks(range(len(PERSONA_LABELS)), PERSONA_LABELS)
plt.title(f"Confusion Matrix (Test Accuracy: {test_acc:.1%})", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Fighting Style")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.show()

# Save Model Checkpoint
checkpoint = {
    "model_state_dict": model.state_dict(),
    "input_dim": len(FEATURE_COLUMNS),
    "hidden_dim": 32,
    "num_classes": NUM_CLASSES,
    "feat_cols": FEATURE_COLUMNS,
    "persona_labels": PERSONA_LABELS,
    "label_to_idx": LABEL_TO_IDX
}
torch.save(checkpoint, "playstyle_classifier.pt")
print("💾 Model checkpoint saved successfully as 'playstyle_classifier.pt'!")


# 🔮 Interactive Fighting Style Predictor
### Predict Any Fighting Style from Battle Actions or Tactical Traits

You can now give **your battle actions, combat turns, or tactical fighting stats**, and the trained model will predict which **Fighting Style Archetype** you belong to.

---

### How to use:
1. **Pass Action Sequence:** Give a list of moves used during combat (e.g. `["Fireball", "Fire Spell", "Quick Slash", ...]`).
2. **Pass Tactical Stat Profile:** Give combat tendencies such as Aggression, Counter-Play frequency, Heal HP threshold, and Decision Speed.
3. The engine extracts the **49 features**, scales them with the fitted scaler, runs forward inference through the neural network, and outputs:
   - 🏆 **Predicted Fighting Style Archetype**
   - 📊 **Confidence Percentage & Probability Distribution**
   - 📈 **Visual Probability Comparison Bar Chart**
   - 💡 **Tactical Archetype Breakdown**


In [ ]:
# ==============================================================================
# 9. MASTER FIGHTING STYLE PREDICTION FUNCTION
# ==============================================================================

def predict_fighting_style(
    actions_or_match: Any = None,
    # Optional tactical fighting style parameters for direct customization:
    aggression_level: Optional[float] = None,       # 0.0 (passive) to 1.0 (ultra-aggressive)
    counter_play_rate: Optional[float] = None,      # 0.0 to 1.0
    heal_hp_threshold: Optional[float] = None,      # 0.1 to 0.8 (fraction of HP when healing is triggered)
    decision_speed_ms: Optional[float] = None,      # reaction time in ms (e.g. 800ms vs 2500ms)
    plot_chart: bool = True
) -> Dict[str, Any]:
    """
    Predicts the fighting style archetype for a player.
    Accepts:
      1. A list of action names (e.g. ['Fireball', 'Fire Spell', 'Quick Slash'])
      2. A full match dictionary conforming to Arcane Clash log schema
      3. Direct tactical fighting traits (aggression, counters, heal threshold, speed)
    """
    # Build match dictionary
    if isinstance(actions_or_match, dict) and "turns" in actions_or_match:
        match_dict = actions_or_match
    elif isinstance(actions_or_match, list) and len(actions_or_match) > 0:
        turns = []
        hero_hp = 110
        power = 50
        for i, act in enumerate(actions_or_match, 1):
            rt = decision_speed_ms if decision_speed_ms is not None else random.randint(1000, 2000)
            turns.append({
                "turn_number": i,
                "action_chosen": act,
                "player_hp_before": hero_hp,
                "player_power_before": power,
                "player_xp_before": 50,
                "damage_dealt": 25 if "Fire" in act else 14,
                "damage_taken": 12,
                "hit_success": True,
                "action_matched_counter": (counter_play_rate is not None and random.random() < counter_play_rate),
                "time_taken_to_decide": rt
            })
            if "Heal" in act: hero_hp = min(110, hero_hp + 30)
            elif "Rest" in act: power = min(100, power + 35)
        match_dict = {"summary": {"hero_class": "Wanderer Magician", "outcome": "win"}, "turns": turns}
    else:
        # Build synthetic profile from high-level traits
        agg = aggression_level if aggression_level is not None else 0.5
        c_rate = counter_play_rate if counter_play_rate is not None else 0.4
        h_thresh = heal_hp_threshold if heal_hp_threshold is not None else 0.35
        speed = decision_speed_ms if decision_speed_ms is not None else 1500.0

        turns = []
        for turn_no in range(1, 15):
            r = random.random()
            if r < agg * 0.6:
                act = random.choice(["Fireball", "Fire Spell"])
            elif r < agg * 0.8:
                act = "Quick Slash"
            elif random.random() < c_rate:
                act = "Water Arrow"
            elif random.random() < 0.2:
                act = "Rest & Recharge"
            else:
                act = "Heal"

            turns.append({
                "turn_number": turn_no,
                "action_chosen": act,
                "player_hp_before": 110 * (h_thresh if act == "Heal" else 0.8),
                "player_power_before": 60 if agg < 0.6 else 20,
                "player_xp_before": 60,
                "damage_dealt": 25 if "Fire" in act else 12,
                "damage_taken": 10 if random.random() < c_rate else 22,
                "hit_success": True,
                "action_matched_counter": (random.random() < c_rate),
                "time_taken_to_decide": speed + random.randint(-200, 200)
            })
        match_dict = {"summary": {"hero_class": "Wanderer Magician", "outcome": "win"}, "turns": turns}

    # Extract features
    raw_feats = extract_match_features(match_dict)

    # Format vector
    x_vec = np.zeros((1, len(FEATURE_COLUMNS)), dtype=np.float32)
    for j, col in enumerate(FEATURE_COLUMNS):
        x_vec[0, j] = float(raw_feats.get(col, 0.0))

    # Scale & infer
    x_scaled = scaler.transform(x_vec)
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        probs = model.predict_proba(x_tensor).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_style = PERSONA_LABELS[pred_idx]
    confidence = float(probs[pred_idx])
    profile = PERSONA_PROFILES[pred_style]

    prob_dict = {label: float(probs[i]) for i, label in enumerate(PERSONA_LABELS)}

    # Rich summary output
    icon_val = profile['icon']
    title_val = profile['title']
    desc_val = profile['description']
    print("=" * 65)
    print(f"🎯 PREDICTED FIGHTING STYLE: {icon_val}  {pred_style.upper()} — '{title_val}'")
    print(f"📊 CONFIDENCE: {confidence:.1%} ({confidence:.4f})")
    print("=" * 65)
    print(f"📝 Profile: {desc_val}")
    print("\n🔍 Key Behavioral Traits Detected:")
    for trait in profile['key_traits']:
        print(f"   • {trait}")
    print("\n📈 Style Probability Breakdown:")
    for style, p in prob_dict.items():
        bar = "█" * int(p * 25)
        print(f"   {style:11s} : {p:6.1%}  {bar}")

    if plot_chart:
        plt.figure(figsize=(7, 3.2))
        colors = [PERSONA_PROFILES[s]["badge_color"] for s in PERSONA_LABELS]
        bars = plt.barh(PERSONA_LABELS, [prob_dict[s] for s in PERSONA_LABELS], color=colors, edgecolor="black", alpha=0.85)
        plt.xlim(0, 1.0)
        plt.xlabel("Model Probability Score", fontsize=10)
        plt.title(f"Fighting Style Inference: {pred_style} ({confidence:.1%})", fontsize=11, fontweight='bold')
        for bar in bars:
            w = bar.get_width()
            plt.text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.1%}", va='center', fontweight='bold', fontsize=9)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    return {
        "fighting_style": pred_style,
        "title": profile["title"],
        "confidence": confidence,
        "probabilities": prob_dict,
        "features": raw_feats
    }

print("✅ Master prediction function 'predict_fighting_style' is ready!")


In [ ]:
# ==============================================================================
# 10. PRESET DEMONSTRATIONS ACROSS ALL FIGHTING STYLES
# ==============================================================================

print("--- [TEST 1: Aggressive Berserker Style] ---")
predict_fighting_style(
    actions_or_match=[
        "Fireball", "Fire Spell", "Fireball", "Fire Spell",
        "Quick Slash", "Fireball", "Fire Spell", "Quick Slash"
    ],
    aggression_level=0.95,
    counter_play_rate=0.1,
    heal_hp_threshold=0.15,
    decision_speed_ms=750
)

print("\n--- [TEST 2: Tactical Guardian Style] ---")
predict_fighting_style(
    actions_or_match=[
        "Rest & Recharge", "Water Arrow", "Heal", "Water Arrow",
        "Rest & Recharge", "Heal", "Rest & Recharge", "Water Arrow"
    ],
    aggression_level=0.1,
    counter_play_rate=0.85,
    heal_hp_threshold=0.60,
    decision_speed_ms=1600
)

print("\n--- [TEST 3: Grand Strategist Style] ---")
predict_fighting_style(
    actions_or_match=[
        "Quick Slash", "Water Arrow", "Rest & Recharge", "Water Arrow",
        "Fireball", "Fire Spell", "Water Arrow", "Heal"
    ],
    aggression_level=0.55,
    counter_play_rate=0.75,
    heal_hp_threshold=0.35,
    decision_speed_ms=2100
)

print("\n--- [TEST 4: Wildcard Sorcerer Style] ---")
predict_fighting_style(
    actions_or_match=[
        "Quick Slash", "Heal", "Fireball", "Rest & Recharge",
        "Water Spell", "Fire Spell", "Water Arrow", "Quick Slash"
    ],
    aggression_level=0.4,
    counter_play_rate=0.3,
    heal_hp_threshold=0.4,
    decision_speed_ms=4500
)


# 🎮 Your Custom Fighting Style Playground
### Input your own battle actions or combat traits below and run this cell to predict your style!


In [ ]:
# ==============================================================================
# 11. YOUR CUSTOM INPUT PLAYGROUND
# ==============================================================================

# Option A: Customize your sequence of actions in battle:
MY_BATTLE_ACTIONS = [
    "Fireball",
    "Fire Spell",
    "Quick Slash",
    "Fireball",
    "Quick Slash",
    "Fire Spell",
    "Rest & Recharge",
    "Fireball"
]

# Option B: Or fine-tune your fighting tendencies (set to None to deduce solely from actions):
MY_AGGRESSION_LEVEL   = 0.90   # 0.0 (Passive/Defensive) to 1.0 (All-out Offense)
MY_COUNTER_PLAY_RATE  = 0.15   # 0.0 (Ignores boss telegraphs) to 1.0 (Always counters)
MY_HEAL_HP_THRESHOLD  = 0.20   # Fraction of max HP (e.g., 0.20 = heals only when below 20% HP)
MY_DECISION_SPEED_MS  = 800    # Milliseconds taken per turn (fast = ~800ms, thoughtful = ~2500ms)

# Run prediction!
result = predict_fighting_style(
    actions_or_match=MY_BATTLE_ACTIONS,
    aggression_level=MY_AGGRESSION_LEVEL,
    counter_play_rate=MY_COUNTER_PLAY_RATE,
    heal_hp_threshold=MY_HEAL_HP_THRESHOLD,
    decision_speed_ms=MY_DECISION_SPEED_MS,
    plot_chart=True
)
